# TP N°02 — Feature Extraction and Embeddings
**Module:** Natural Language Processing  
**Option:** IASD — 2ème année Second Cycle  
**Prepared by:** Dr R. BOUSMAHA

---
This notebook covers **Sections A through F**: Data preparation, encoding, train/test split, vectorization methods (Bag-of-Words, One-Hot, TF-IDF), model training, evaluation, and timing.

## 🔧 Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn — preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Sklearn — vectorization
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Sklearn — models
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Sklearn — evaluation
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All libraries imported successfully.")

---
## A. 📂 Data Preparation
Import the preprocessed `spooky.csv` dataset (final version after preprocessing from TP01).

In [ ]:
# Load the preprocessed spooky dataset
# Make sure 'spooky.csv' is in your working directory
# The dataset should already be preprocessed (lowercased, stopwords removed, lemmatized, etc.)

df = pd.read_csv('spooky.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Inspect columns and data types
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# The text column (preprocessed) and label column
# Adjust column names to match your actual dataset
# Common setup: 'text' or 'cleaned_text' for text, 'author' for labels

TEXT_COL = 'text'    # Change if your column has a different name
LABEL_COL = 'author' # Change if your label column has a different name

# Drop rows with missing values
df = df[[TEXT_COL, LABEL_COL]].dropna()

print(f"Dataset size after cleaning: {df.shape}")
print(f"\nClass distribution:")
print(df[LABEL_COL].value_counts())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(7, 4))
df[LABEL_COL].value_counts().plot(kind='bar', color=['steelblue','salmon','mediumseagreen'])
plt.title('Class Distribution (Authors)')
plt.xlabel('Author')
plt.ylabel('Number of samples')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n⚠️  The dataset is imbalanced — we will use stratified splitting.")

---
## B. 🔢 Encoding the Target Variable
Encode the author labels using `LabelEncoder`.

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df[LABEL_COL])

print("Label mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls} → {i}")

print("\nSample encoded labels:")
print(df[['author', 'label']].head(10))

In [ ]:
# Extract features (X) and target (y)
X = df[TEXT_COL].values  # Raw preprocessed text
y = df['label'].values   # Encoded integer labels

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Classes: {np.unique(y)}")

---
## C. 📊 Train / Test Split
Split the dataset (70% train / 30% test) with **stratification** to handle class imbalance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=0,
    stratify=y   # ← stratification to preserve class proportions
)

print(f"Training set size  : {len(X_train)} samples")
print(f"Test set size      : {len(X_test)} samples")
print(f"\nClass distribution in train: {np.bincount(y_train)}")
print(f"Class distribution in test : {np.bincount(y_test)}")

In [ ]:
# Verify stratification proportions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts  = pd.Series(y_test).value_counts().sort_index()

axes[0].bar(le.classes_, train_counts.values, color='steelblue')
axes[0].set_title('Train Set Distribution')
axes[0].set_xlabel('Author')
axes[0].set_ylabel('Count')

axes[1].bar(le.classes_, test_counts.values, color='salmon')
axes[1].set_title('Test Set Distribution')
axes[1].set_xlabel('Author')
axes[1].set_ylabel('Count')

plt.suptitle('Stratified Split — Class Proportions Preserved')
plt.tight_layout()
plt.show()

---
## D. 🔤 Vectorization Methods

We apply three vectorization approaches:
1. **Bag-of-Words (CountVectorizer)** — term frequency / lexical frequency
2. **One-Hot Encoding** — binary presence/absence of each word
3. **TF-IDF** — Term Frequency–Inverse Document Frequency

### D.1 — Bag-of-Words (Lexical Frequency)

In [ ]:
# Bag-of-Words: counts how many times each word appears
bow_vectorizer = CountVectorizer(max_features=10000)

# Fit on training set ONLY, then transform both
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow  = bow_vectorizer.transform(X_test)

print("BoW — Train shape:", X_train_bow.shape)
print("BoW — Test shape :", X_test_bow.shape)
print("\nSample vocabulary (first 20 words):")
print(bow_vectorizer.get_feature_names_out()[:20])

### D.1b — One-Hot Encoding

In [ ]:
# One-Hot Encoding: binary — 1 if word present, 0 otherwise (binary=True)
ohe_vectorizer = CountVectorizer(max_features=10000, binary=True)

X_train_ohe = ohe_vectorizer.fit_transform(X_train)
X_test_ohe  = ohe_vectorizer.transform(X_test)

print("OHE — Train shape:", X_train_ohe.shape)
print("OHE — Test shape :", X_test_ohe.shape)

# Show difference: BoW vs OHE on same sentence
sample = ["the cat sat on the mat"]
tiny_bow = CountVectorizer()
tiny_ohe = CountVectorizer(binary=True)
bow_ex = tiny_bow.fit_transform(sample).toarray()
ohe_ex = tiny_ohe.fit_transform(sample).toarray()
print("\n--- Illustration on 'the cat sat on the mat' ---")
print("Vocab :", tiny_bow.get_feature_names_out())
print("BoW   :", bow_ex[0])
print("OHE   :", ohe_ex[0])

### D.2 — TF-IDF Vectorization

In [ ]:
# TF-IDF: balances term frequency with inverse document frequency
tfidf_vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)

# D.2: Fit on TRAIN only
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# D.3: Use SAME model to transform TEST
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print("TF-IDF — Train shape:", X_train_tfidf.shape)
print("TF-IDF — Test shape :", X_test_tfidf.shape)

# Top TF-IDF words
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_tfidf    = X_train_tfidf.mean(axis=0).A1
top_indices   = mean_tfidf.argsort()[-15:][::-1]
print("\nTop 15 words by mean TF-IDF score:")
for idx in top_indices:
    print(f"  {feature_names[idx]:<20} {mean_tfidf[idx]:.4f}")

In [ ]:
# Summary of all vectorized representations
print("=" * 50)
print(" VECTORIZATION SUMMARY")
print("=" * 50)
print(f"{'Method':<15} {'Train Shape':<20} {'Test Shape':<20}")
print("-" * 55)
print(f"{'BoW':<15} {str(X_train_bow.shape):<20} {str(X_test_bow.shape):<20}")
print(f"{'One-Hot':<15} {str(X_train_ohe.shape):<20} {str(X_test_ohe.shape):<20}")
print(f"{'TF-IDF':<15} {str(X_train_tfidf.shape):<20} {str(X_test_tfidf.shape):<20}")

---
## E. 🧠 Model Training
Train three **MLPClassifier** models — one per vectorization representation.

In [ ]:
# Helper function to print classification report + confusion matrix
def evaluate_model(model_name, y_true, y_pred, classes):
    """Print classification report and plot confusion matrix."""
    print(f"\n{'='*55}")
    print(f" Classification Report — {model_name}")
    print(f"{'='*55}")
    print(classification_report(y_true, y_pred, target_names=classes))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(f'Confusion Matrix — {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Model 1: MLP on Bag-of-Words ──────────────────────────
print("Training Model 1: MLP + Bag-of-Words ...")
t0 = time.time()

mlp_bow = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42,
    early_stopping=True,
    verbose=False
)
mlp_bow.fit(X_train_bow, y_train)
time_bow = time.time() - t0

print(f"✅ Model 1 trained in {time_bow:.2f}s")

In [ ]:
# ── Model 2: MLP on One-Hot Encoding ──────────────────────
print("Training Model 2: MLP + One-Hot Encoding ...")
t0 = time.time()

mlp_ohe = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42,
    early_stopping=True,
    verbose=False
)
mlp_ohe.fit(X_train_ohe, y_train)
time_ohe = time.time() - t0

print(f"✅ Model 2 trained in {time_ohe:.2f}s")

In [ ]:
# ── Model 3: MLP on TF-IDF ────────────────────────────────
print("Training Model 3: MLP + TF-IDF ...")
t0 = time.time()

mlp_tfidf = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42,
    early_stopping=True,
    verbose=False
)
mlp_tfidf.fit(X_train_tfidf, y_train)
time_tfidf = time.time() - t0

print(f"✅ Model 3 trained in {time_tfidf:.2f}s")

In [ ]:
# E.3 — Predict on TRAINING set (to check for overfitting)
y_pred_train_bow   = mlp_bow.predict(X_train_bow)
y_pred_train_ohe   = mlp_ohe.predict(X_train_ohe)
y_pred_train_tfidf = mlp_tfidf.predict(X_train_tfidf)

# E.4 — Training classification reports
print("\n" + "#"*60)
print("#  SECTION E — TRAINING SET PERFORMANCE")
print("#"*60)

evaluate_model("MLP + BoW  [TRAIN]",   y_train, y_pred_train_bow,   le.classes_)
evaluate_model("MLP + OHE  [TRAIN]",   y_train, y_pred_train_ohe,   le.classes_)
evaluate_model("MLP + TFIDF [TRAIN]",  y_train, y_pred_train_tfidf, le.classes_)

---
## F. 🧪 Testing & Evaluation
Apply the three trained models on the **test set** and compute metrics + training times.

In [ ]:
# F.1 — Predict on TEST set
y_pred_test_bow   = mlp_bow.predict(X_test_bow)
y_pred_test_ohe   = mlp_ohe.predict(X_test_ohe)
y_pred_test_tfidf = mlp_tfidf.predict(X_test_tfidf)

# F.2 — Test classification reports
print("\n" + "#"*60)
print("#  SECTION F — TEST SET PERFORMANCE")
print("#"*60)

evaluate_model("MLP + BoW  [TEST]",   y_test, y_pred_test_bow,   le.classes_)
evaluate_model("MLP + OHE  [TEST]",   y_test, y_pred_test_ohe,   le.classes_)
evaluate_model("MLP + TFIDF [TEST]",  y_test, y_pred_test_tfidf, le.classes_)

In [ ]:
# F.3 — Training time comparison
print("\n" + "="*45)
print(" TRAINING TIME COMPARISON")
print("="*45)
print(f"{'Model':<25} {'Training Time (s)':<20}")
print("-"*45)
print(f"{'MLP + BoW':<25} {time_bow:.2f}s")
print(f"{'MLP + One-Hot':<25} {time_ohe:.2f}s")
print(f"{'MLP + TF-IDF':<25} {time_tfidf:.2f}s")

# Bar chart for training times
plt.figure(figsize=(7, 4))
models  = ['MLP + BoW', 'MLP + OHE', 'MLP + TF-IDF']
times   = [time_bow, time_ohe, time_tfidf]
colors  = ['steelblue', 'salmon', 'mediumseagreen']
plt.bar(models, times, color=colors, edgecolor='black')
plt.title('Training Time per Model')
plt.ylabel('Time (seconds)')
plt.xlabel('Model')
for i, v in enumerate(times):
    plt.text(i, v + 0.1, f'{v:.1f}s', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Global accuracy comparison — all models on test set
acc_bow   = accuracy_score(y_test, y_pred_test_bow)
acc_ohe   = accuracy_score(y_test, y_pred_test_ohe)
acc_tfidf = accuracy_score(y_test, y_pred_test_tfidf)

print("\n" + "="*45)
print(" TEST ACCURACY SUMMARY")
print("="*45)
print(f"{'Model':<25} {'Accuracy':<15}")
print("-"*40)
print(f"{'MLP + BoW':<25} {acc_bow:.4f}")
print(f"{'MLP + One-Hot':<25} {acc_ohe:.4f}")
print(f"{'MLP + TF-IDF':<25} {acc_tfidf:.4f}")

best = max([(acc_bow, 'MLP + BoW'), (acc_ohe, 'MLP + OHE'), (acc_tfidf, 'MLP + TF-IDF')])
print(f"\n🏆 Best model: {best[1]} with accuracy = {best[0]:.4f}")

In [ ]:
# Summary bar chart — accuracy comparison
plt.figure(figsize=(7, 4))
accuracies = [acc_bow, acc_ohe, acc_tfidf]
bars = plt.bar(models, accuracies, color=colors, edgecolor='black')
plt.ylim(0, 1.05)
plt.title('Test Accuracy per Vectorization Method')
plt.ylabel('Accuracy')
plt.xlabel('Model')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📊 Full Comparison Report (A → F)
Consolidated comparison of all three models across all metrics.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def get_metrics(y_true, y_pred, model_name, train_time):
    return {
        'Model'     : model_name,
        'Accuracy'  : round(accuracy_score(y_true, y_pred), 4),
        'Precision' : round(precision_score(y_true, y_pred, average='weighted'), 4),
        'Recall'    : round(recall_score(y_true, y_pred, average='weighted'), 4),
        'F1-Score'  : round(f1_score(y_true, y_pred, average='weighted'), 4),
        'Train Time': f"{train_time:.2f}s"
    }

results = [
    get_metrics(y_test, y_pred_test_bow,   'MLP + BoW',    time_bow),
    get_metrics(y_test, y_pred_test_ohe,   'MLP + OHE',    time_ohe),
    get_metrics(y_test, y_pred_test_tfidf, 'MLP + TF-IDF', time_tfidf),
]

results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
print("\n📋 FINAL COMPARISON TABLE (Test Set)")
print("="*70)
print(results_df.to_string())
print("="*70)

In [ ]:
# Multi-metric comparison chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, (model_name, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=model_name, color=colors[i], edgecolor='black')

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — BoW vs One-Hot vs TF-IDF (Test Set)')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()

print("\n✅ TP02 Sections A–F complete!")

---
## 📝 Conclusion

| Section | Task | Status |
|---------|------|--------|
| A | Load preprocessed spooky.csv | ✅ |
| B | Label encoding (LabelEncoder) | ✅ |
| C | Stratified train/test split (70/30, random_state=0) | ✅ |
| D.1 | Bag-of-Words + One-Hot vectorization | ✅ |
| D.2 | TF-IDF fitted on train | ✅ |
| D.3 | TF-IDF applied to test (same model) | ✅ |
| E | 3× MLP trained, predictions on train, classification reports | ✅ |
| F | Predictions on test, reports, training times compared | ✅ |

### Key Observations:
- **TF-IDF** generally outperforms raw BoW/OHE as it down-weights common uninformative words.
- **One-Hot** loses term frequency information, which can hurt performance.
- **Training time** depends on the density and dimensionality of the feature space.
- **Stratified splitting** ensures fair evaluation despite the class imbalance in the Spooky dataset.

> **Next:** Section G will explore word embeddings (Word2Vec, GloVe, FastText) for richer representations.